Testing monthly compliance

In [6]:
# optimized_monthly_compliance.py
import os
import shutil
import logging
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import gc
from openpyxl import load_workbook
from openpyxl.worksheet.table import Table, TableStyleInfo

# ---------- CONFIG ----------
folder_path = os.environ["FOLDER_PATH_gme_compliance"]
old_file_folder = 'past_lists'
old_file_folder_path = os.path.join(folder_path, old_file_folder)

PILOT_ONLY = True
PILOTS = ['NEUROSURG-Neurological Surgery-ACGME', 'Imaging-Diagnostic Radiology-ACGME', 'MED-Pulmonary Disease & Critical Care Medicine-ACGME',
          'RAD-Radiation Oncology-ACGME', 'PEDS-Pediatric Medicine-ACGME', 'Surgery-Advanced GI MIS/Bariatric', 'MED-Hospice & Palliative Care Medicine-ACGME',
          'OB/GYN-Obstetrics & Gynecology-ACGME', 'MED-Rheumatology-ACGME', 'MED-Rheumatology-Research']
OUTPUT_PREFIX = "monthly_compliance_email_list"

HOURS_COLS_NEW = ["Person's National Provider Identifier", 'Person', 'Status', 'Program',
       'Work Type', 'Actual Start', 'Actual End', 'Actual Hours Worked',
       'Rotation', 'Rotation Start Date', 'Rotation End Date', 'Source',
       'Resident Approved', 'Administrator Approved', 'Institution/Location',
       'In Violation', 'Violation(s)', 'Rules Violated', 'Comment', 'Comment By',
       'Last Update', "Date Logged", "Program Admin Email",
       "Trainee Email", "Person's Program Coordinator",
       "Person's Program Director", 'Trainee Last Name', 'Trainee First Name']

ACTIVE_COLS_NEW = ['ID Number', 'Trainee Last Name', 'Trainee First Name', 'Middle Name',
       "Person's National Provider Identifier",
       "Trainee Email", 'Department/Division', 'Program',
       "Person's Program Director", 'Status', "Person's Program Start Date",
       "Person's Program End Date", "Program Admin Email",
       "Person's Program Coordinator"]

PD_LIST_COLS_NEW = ['Program', 'programtype', 'department', 'Program Director First Name',
       'Program Director Last Name', 'programdirector', 'Program Director Email',
       'programcoordinator', 'Program Admin Email']

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

# ---------- Utilities ----------
def ensure_dirs():
    os.makedirs(old_file_folder_path, exist_ok=True)
    os.makedirs(os.path.join(old_file_folder_path, 'old_active_list'), exist_ok=True)
    os.makedirs(os.path.join(old_file_folder_path, 'old_hours_list'), exist_ok=True)
    os.makedirs(os.path.join(folder_path, 'past_lists', 'old_compliance_list'), exist_ok=True)

def read_inputs():
    active = pd.read_excel(os.path.join(folder_path, 'active.xlsx'))
    hours = pd.read_excel(os.path.join(folder_path, 'hours.xlsx'))
    pd_list = pd.read_excel(os.path.join(folder_path, 'PD_and_PA_report_list.xlsx'))
    return active, hours, pd_list

def normalize_and_clean(active, hours, pd_list):
    # Split 'Person' into first/last names for hours
    hours[['Trainee Last Name', 'Trainee First Name']] = hours['Person'].str.split(',', n=1, expand=True)
    hours['Trainee Last Name'] = hours['Trainee Last Name'].str.strip()
    hours['Trainee First Name'] = hours['Trainee First Name'].str.strip()
    
    # Rename columns
    hours.columns = HOURS_COLS_NEW
    active.columns = ACTIVE_COLS_NEW
    pd_list.columns = PD_LIST_COLS_NEW

    # Lowercase emails
    for df, col in [(hours, 'Trainee Email'), (active, 'Trainee Email'),
                    (hours, 'Program Admin Email'), (active, 'Program Admin Email'),
                    (pd_list, 'Program Admin Email')]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.lower().replace({'nan': np.nan})

    # Remove Chief Residents
    if 'Status' in active.columns:
        active = active[active['Status'] != 'Chief Resident']

    # Parse datetimes
    for c in ['Actual Start', 'Actual End', 'Date Logged', 'Last Update']:
        if c in hours.columns:
            hours[c] = pd.to_datetime(hours[c], errors='coerce')
    return active, hours, pd_list

def prev_month_range(reference_date=None):
    if reference_date is None:
        reference_date = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0)
    first_of_this_month = reference_date.replace(day=1)
    end_last_month = first_of_this_month - timedelta(days=1)
    start_last_month = end_last_month.replace(day=1)
    return start_last_month, end_last_month


def generate_full_weeks_for_month(start_month, end_month):
    """
    Generates weekly periods that OVERLAP the target month.

    Includes:
    - weeks starting before the month
    - weeks ending after the month
    - partial weeks at beginning or end

    Excludes:
    - weeks fully outside the month
    """

    month_start = start_month.replace(day=1)
    month_end = end_month

    # Find the Sunday on or before month_start
    first_sunday = month_start - timedelta(days=(month_start.weekday() + 1) % 7)

    weeks = []
    current_start = first_sunday

    while current_start <= month_end:
        current_end = current_start + timedelta(days=7)

        # ✅ KEEP WEEK IF IT OVERLAPS THE MONTH
        if current_start <= month_end and current_end >= month_start:
            week_label = (
                f"{current_start.strftime('%Y-%m-%d')} "
                f"to {current_end.strftime('%Y-%m-%d')}"
            )
            weeks.append((current_start, current_end, week_label))

        current_start += timedelta(days=7)

    return weeks


# ---------- Core Processing ----------
def process_month(active, hours, pd_list, start_month, end_month):
    active = active.copy()
    active['Trainee Email'] = active['Trainee Email'].str.lower().str.strip()
    hours['Trainee Email'] = hours['Trainee Email'].str.lower().str.strip()
    active_emails = set(active['Trainee Email'].dropna())

    trainee_info = {}
    violations_map = {}
    resq_map = {}
    missing_weeks_map = {}

    # Pre-fill trainee info from active
    for _, row in active.iterrows():
        email = row.get('Trainee Email')
        if pd.isna(email):
            continue
        trainee_info[email] = {
            'Trainee First Name': row.get('Trainee First Name'),
            'Trainee Last Name': row.get('Trainee Last Name'),
            'Program': row.get('Program'),
            'Program Admin Email': row.get('Program Admin Email')
        }

    weeks = generate_full_weeks_for_month(start_month, end_month)

    for ws, we, week_label in weeks:
        mask = (hours['Actual Start'] >= ws) & (hours['Actual Start'] <= we)
        hours_week = hours.loc[mask].copy()

        # RESQ detection
        resq_entries = hours_week[hours_week['Work Type'].str.contains('ResQ', na=False, case=False)]
        for _, r in resq_entries.iterrows():
            email = r.get('Trainee Email')
            if pd.isna(email): continue
            resq_map[email] = True
            if email not in trainee_info:
                trainee_info[email] = {
                    'Trainee First Name': r.get('Trainee First Name'),
                    'Trainee Last Name': r.get('Trainee Last Name'),
                    'Program': r.get('Program'),
                    'Program Admin Email': r.get('Program Admin Email')
                }

        # Violations detection
        if 'In Violation' in hours_week.columns:
            inv_series = hours_week['In Violation'].astype(str).str.strip().str.lower()
            valid_yes = inv_series.isin(['yes','y'])
            violations_entries = hours_week.loc[valid_yes]
            for _, v in violations_entries.iterrows():
                email = v.get('Trainee Email')
                if pd.isna(email): continue
                viol_msg = f"{v.get('Actual Start').strftime('%m/%d/%Y') if pd.notna(v.get('Actual Start')) else ''} {v.get('Rules Violated','')}"
                violations_map.setdefault(email, set()).add(viol_msg.strip())
                if email not in trainee_info:
                    trainee_info[email] = {
                        'Trainee First Name': v.get('Trainee First Name'),
                        'Trainee Last Name': v.get('Trainee Last Name'),
                        'Program': v.get('Program'),
                        'Program Admin Email': v.get('Program Admin Email')
                    }

        # Missing hours: no entries
        emails_this_week = set(hours_week['Trainee Email'].dropna())
        no_entry_emails = active_emails - emails_this_week
        for email in no_entry_emails:
            missing_weeks_map.setdefault(email, set()).add(week_label)

        # Partial coverage (<4 days)
        if not hours_week.empty:
            def expand_shift_days(row):
                s, e = row['Actual Start'], row['Actual End']
                if pd.isna(s) or pd.isna(e): return []
                start_date, end_date = s.date(), e.date()
                if end_date < start_date: return [start_date]
                return list(pd.date_range(start_date, end_date).date)
            hours_week['Days Covered'] = hours_week.apply(expand_shift_days, axis=1)
            df_days = hours_week.explode('Days Covered')
            days_worked = df_days.groupby('Trainee Email')['Days Covered'].nunique().reset_index()
            partials = days_worked[days_worked['Days Covered'] < 4]
            for _, p in partials.iterrows():
                email = p['Trainee Email']
                missing_weeks_map.setdefault(email, set()).add(week_label)

    # Build final DataFrame — only include trainees who have at least one issue
    all_emails = set(trainee_info.keys()) | set(violations_map.keys()) | set(resq_map.keys()) | set(missing_weeks_map.keys())

    # Keep only emails that actually have a problem
    def has_issue(email):
        if email in resq_map and resq_map.get(email, False):
            return True
        if email in violations_map and violations_map.get(email):
            return True
        if email in missing_weeks_map and missing_weeks_map.get(email):
            return True
        return False

    filtered_emails = {e for e in all_emails if e is not None and not pd.isna(e) and has_issue(e)}

    rows = []
    for email in sorted(filtered_emails):
        info = trainee_info.get(email, {})
        rows.append({
            'Trainee Email': email,
            'Trainee First Name': info.get('Trainee First Name'),
            'Trainee Last Name': info.get('Trainee Last Name'),
            'Program': info.get('Program'),
            'Program Admin Email': info.get('Program Admin Email'),
            'ResQ Violations': 'Yes' if resq_map.get(email, False) else np.nan,
            'Violations': ', '.join(sorted(violations_map.get(email, []))) if violations_map.get(email) else np.nan,
            'Week(s) of Missing Hours': ', '.join(sorted(missing_weeks_map.get(email, []))) if missing_weeks_map.get(email) else np.nan
        })

    consolidated_df = pd.DataFrame(rows)

    # Optional pilot filter
    if PILOT_ONLY:
        consolidated_df = consolidated_df[consolidated_df['Program'].isin(PILOTS)]

    return consolidated_df



# creating a weekly generator of df based on date input

this is used to compare against the monthly compliance generator

In [7]:
def get_week_bounds(input_date):
    """
    Given a date, returns the Sunday–Saturday week containing that date.
    """

    if not isinstance(input_date, datetime):
        input_date = pd.to_datetime(input_date)

    input_date = input_date.replace(hour=0, minute=0, second=0, microsecond=0)

    # Sunday = 6 → convert
    days_since_sunday = (input_date.weekday() + 1) % 7

    week_start = input_date - timedelta(days=days_since_sunday)
    week_end = week_start + timedelta(days=6)

    return (
        week_start,
        week_end.replace(hour=23, minute=59, second=59)
    )


In [8]:
def build_compliance_df(
    week_date,
    folder_path
):
    """
    Builds weekly compliance dataframe for the week containing `week_date`.

    Parameters
    ----------
    week_date : str | datetime
        Any date within the week of interest.
    folder_path : str
        Folder containing active.xlsx, hours.xlsx, PD_and_PA_report_list.xlsx

    Returns
    -------
    consolidated_df1 : pd.DataFrame
    """

    import os
    import pandas as pd

    # -----------------------------
    # Load data
    # -----------------------------
    active = pd.read_excel(os.path.join(folder_path, "active.xlsx"))
    hours = pd.read_excel(os.path.join(folder_path, "hours.xlsx"))
    pd_list = pd.read_excel(os.path.join(folder_path, "PD_and_PA_report_list.xlsx"))

    # -----------------------------
    # Week bounds from input date
    # -----------------------------
    start_of_week, end_of_week = get_week_bounds(week_date)

    # -----------------------------
    # Name parsing
    # -----------------------------
    hours[['Trainee Last Name', 'Trainee First Name']] = (
        hours['Person']
        .str.split(',', n=1, expand=True)
    )

    hours['Trainee Last Name'] = hours['Trainee Last Name'].str.strip()
    hours['Trainee First Name'] = hours['Trainee First Name'].str.strip()

    # -----------------------------
    # Column renames
    # -----------------------------
    hours.columns = [
        "Person's National Provider Identifier", 'Person', 'Status', 'Program',
        'Work Type', 'Actual Start', 'Actual End', 'Actual Hours Worked',
        'Rotation', 'Rotation Start Date', 'Rotation End Date', 'Source',
        'Resident Approved', 'Administrator Approved', 'Institution/Location',
        'In Violation', 'Violation(s)', 'Rules Violated', 'Comment', 'Comment By',
        'Last Update', "Date Logged", "Program Admin Email",
        "Trainee Email", "Person's Program Coordinator",
        "Person's Program Director", 'Trainee Last Name',
        'Trainee First Name'
    ]

    active.columns = [
        'ID Number', 'Trainee Last Name', 'Trainee First Name',
        'Middle Name',
        "Person's National Provider Identifier",
        "Trainee Email", 'Department/Division', 'Program',
        "Person's Program Director", 'Status',
        "Person's Program Start Date",
        "Person's Program End Date",
        "Program Admin Email",
        "Person's Program Coordinator"
    ]

    # -----------------------------
    # Cleanup
    # -----------------------------
    hours['Trainee Email'] = hours['Trainee Email'].str.lower()
    active['Trainee Email'] = active['Trainee Email'].str.lower()

    hours['Program Admin Email'] = hours['Program Admin Email'].str.lower()
    active['Program Admin Email'] = active['Program Admin Email'].str.lower()

    pd_list['programcoordinatoremail'] = pd_list['programcoordinatoremail'].str.lower()

    active = active[active['Status'] != 'Chief Resident']

    pd_list.columns = [
        'Program', 'programtype', 'department',
        'Program Director First Name',
        'Program Director Last Name',
        'programdirector',
        'Program Director Email',
        'programcoordinator',
        'Program Admin Email'
    ]

    # -----------------------------
    # Filter week of interest
    # -----------------------------
    mask_hours = (
        (hours['Actual End'] >= start_of_week) &
        (hours['Actual Start'] <= end_of_week)
    )

    df_week = hours.loc[mask_hours].copy()

    mask_non_hours = (
        (hours['Actual Start'] >= start_of_week) &
        (hours['Actual Start'] <= end_of_week)
    )

    df_week_non_hours = hours.loc[mask_non_hours].copy()

    # -----------------------------
    # ResQ
    # -----------------------------
    resQ = df_week_non_hours[df_week_non_hours['Work Type'] == 'ResQ Working']

    consolidated_resQ = (
        resQ.groupby('Trainee Email', as_index=False)
        .agg({
            'Trainee First Name': 'first',
            'Trainee Last Name': 'first',
            'Program Admin Email': 'first',
            'Program': 'first'
        })
    )

    consolidated_resQ['ResQ Violations'] = 'Yes'

    # -----------------------------
    # Violations
    # -----------------------------
    df_week_non_hours['In Violation'] = (
        df_week_non_hours['In Violation']
        .astype(str).str.strip().str.lower()
    )

    violations = df_week_non_hours[df_week_non_hours['In Violation'].isin(['yes', 'y'])]

    violations['Violations'] = (
        violations['Actual Start'].dt.strftime('%m/%d/%Y') + ' ' + violations['Rules Violated']
    )

    consolidated_violations = (
        violations.groupby('Trainee Email', as_index=False)
        .agg({
            'Trainee First Name': 'first',
            'Trainee Last Name': 'first',
            'Program Admin Email': 'first',
            'Program': 'first',
            'Violations': lambda x: ', '.join(sorted(set(x)))
        })
    )

    # -----------------------------
    # Missing hours (update: also count if <4 entries)
    # -----------------------------
    hours_count = df_week.groupby('Trainee Email').size().reset_index(name='Hours Count')

    active_emails = active['Trainee Email'].unique()
    hours_count_dict = dict(zip(hours_count['Trainee Email'], hours_count['Hours Count']))

    # Flag trainees with <4 entries (includes those with 0)
    missing_or_insufficient = [email for email in active_emails if hours_count_dict.get(email, 0) < 4]

    consolidated_missing = (
        active[active['Trainee Email'].isin(missing_or_insufficient)]
        .groupby('Trainee Email', as_index=False)
        .agg({
            'Trainee First Name': 'first',
            'Trainee Last Name': 'first',
            'Program': 'first',
            'Program Admin Email': 'first'
        })
    )

    consolidated_missing['Week of Missing Hours'] = (
        start_of_week.strftime('%m/%d/%Y') + ' - ' + end_of_week.strftime('%m/%d/%Y')
    )

    # -----------------------------
    # Combine everything
    # -----------------------------
    table = pd.concat(
        [consolidated_resQ, consolidated_missing, consolidated_violations],
        ignore_index=True
    )

    consolidated = table.groupby('Trainee Email', as_index=False).first()

    consolidated = consolidated.merge(
        pd_list[[
            'Program Admin Email',
            'Program',
            'Program Director First Name',
            'Program Director Last Name',
            'Program Director Email'
        ]],
        on=['Program Admin Email', 'Program'],
        how='left'
    )

    return consolidated


In [9]:
pilots = ['NEUROSURG-Neurological Surgery-ACGME', 'Imaging-Diagnostic Radiology-ACGME', 'MED-Pulmonary Disease & Critical Care Medicine-ACGME',
          'RAD-Radiation Oncology-ACGME', 'PEDS-Pediatric Medicine-ACGME', 'Surgery-Advanced GI MIS/Bariatric', 'MED-Hospice & Palliative Care Medicine-ACGME',
          'OB/GYN-Obstetrics & Gynecology-ACGME', 'MED-Rheumatology-ACGME', 'MED-Rheumatology-Research']


In [10]:
import os
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from openpyxl import load_workbook
import numpy as np

import ast

import requests
import shutil

import logging
import sys
import re
from pathlib import Path
from datetime import datetime, timedelta
from openpyxl import load_workbook
from openpyxl.worksheet.table import Table, TableStyleInfo

In [11]:
df1 = build_compliance_df(
    week_date="2026-03-01",
    folder_path=os.environ["FOLDER_PATH_gme_compliance"]
)

df1 = df1[df1['Program'].isin(pilots)]

C:\Users\MartinezB11\AppData\Local\Temp\ipykernel_27560\2880171935.py:140: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  violations['Violations'] = (


In [12]:
df2 = build_compliance_df(
    week_date="2026-03-08",
    folder_path=os.environ["FOLDER_PATH_gme_compliance"]
)

df2 = df2[df2['Program'].isin(pilots)]

C:\Users\MartinezB11\AppData\Local\Temp\ipykernel_27560\2880171935.py:140: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  violations['Violations'] = (


In [13]:
df3 = build_compliance_df(
    week_date="2026-03-15",
    folder_path=os.environ["FOLDER_PATH_gme_compliance"]
)

df3 = df3[df3['Program'].isin(pilots)]

C:\Users\MartinezB11\AppData\Local\Temp\ipykernel_27560\2880171935.py:140: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  violations['Violations'] = (


In [14]:
df4 = build_compliance_df(
    week_date="2026-03-21",
    folder_path=os.environ["FOLDER_PATH_gme_compliance"]
)

df4 = df4[df4['Program'].isin(pilots)]

C:\Users\MartinezB11\AppData\Local\Temp\ipykernel_27560\2880171935.py:140: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  violations['Violations'] = (


In [15]:
df5 = build_compliance_df(
    week_date="2026-03-28",
    folder_path=os.environ["FOLDER_PATH_gme_compliance"]
)

df5 = df5[df5['Program'].isin(pilots)]

C:\Users\MartinezB11\AppData\Local\Temp\ipykernel_27560\2880171935.py:140: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  violations['Violations'] = (


In [18]:
check_dep = ['Imaging-Diagnostic Radiology-ACGME', 'RAD-Radiation Oncology-ACGME', 'Surgery-Advanced GI MIS/Bariatric', 'MED-Rheumatology-ACGME', 'MED-Rheumatology-Research']

In [28]:
df5

,Trainee Email,Trainee First Name,Trainee Last Name,Program Admin Email,Program,ResQ Violations,Week of Missing Hours,Violations,Program Director First Name,Program Director Last Name,Program Director Email
36,catherine.garcia@cshs.org,Catherine Michelle,Garcia,samantha.phu@cshs.org,NEUROSURG-Neurological Surgery-ACGME,Yes,None,None,Chirag,Patil,patilc@cshs.org
61,edwina.tran@cshs.org,Edwina Beryl,Tran,samantha.phu@cshs.org,NEUROSURG-Neurological Surgery-ACGME,Yes,None,None,Chirag,Patil,patilc@cshs.org
67,enrique.vargas@cshs.org,Enrique,Vargas,samantha.phu@cshs.org,NEUROSURG-Neurological Surgery-ACGME,None,03/22/2026 - 03/28/2026,None,Chirag,Patil,patilc@cshs.org
85,irene.masini@cshs.org,Irene,Masini,morgan.mckay@cshs.org,OB/GYN-Obstetrics & Gynecology-ACGME,None,03/22/2026 - 03/28/2026,None,Gabriela,Dellapiana,gabriela.dellapiana@cshs.org
129,lauren.hosek@cshs.org,Lauren,Hosek,morgan.mckay@cshs.org,OB/GYN-Obstetrics & Gynecology-ACGME,None,03/22/2026 - 03/28/2026,None,Gabriela,Dellapiana,gabriela.dellapiana@cshs.org
148,megan.macdougall@cshs.org,Megan Riley,Macdougall,brittany.smith@cshs.org,MED-Pulmonary Disease & Critical Care Medicine...,None,None,"03/25/2026 ACGME Short Break, 03/27/2026 ACGME...",Jeremy,Falk,jeremy.falk@cshs.org
156,micah.devalle@cshs.org,Micah K.,de Valle,morgan.mckay@cshs.org,OB/GYN-Obstetrics & Gynecology-ACGME,Yes,03/22/2026 - 03/28/2026,None,Gabriela,Dellapiana,gabriela.dellapiana@cshs.org
171,neil.larson@cshs.org,Neil Patrick,Larson,christina.bussell@cshs.org,MED-Hospice & Palliative Care Medicine-ACGME,None,None,03/22/2026 ACGME Day Off,Azadeh,Dashti,azadeh.dashti@cshs.org
178,olivia.foy@cshs.org,Olivia,Foy,morgan.mckay@cshs.org,OB/GYN-Obstetrics & Gynecology-ACGME,None,03/22/2026 - 03/28/2026,None,Gabriela,Dellapiana,gabriela.dellapiana@cshs.org
205,shane.shahrestani@cshs.org,Shane,Shahrestani,samantha.phu@cshs.org,NEUROSURG-Neurological Surgery-ACGME,Yes,None,None,Chirag,Patil,patilc@cshs.org


In [19]:
df1['Program'].isin(check_dep).value_counts()

Program
False    9
Name: count, dtype: int64

In [20]:
df2['Program'].isin(check_dep).value_counts()

Program
False    9
Name: count, dtype: int64

In [21]:
df3['Program'].isin(check_dep).value_counts()


Program
False    13
Name: count, dtype: int64

In [22]:
df4['Program'].isin(check_dep).value_counts()

Program
False    13
Name: count, dtype: int64